# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveenadanthapally/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
# ============================================================
# WEEK 6 — LOAD THE REAL WEEK-5 DATASET
# ============================================================

import pandas as pd
import numpy as np

DATA_URL = "https://raw.githubusercontent.com/praveenadanthapally/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

print("REAL DATA LOADED")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumn names:")
print(df.columns.tolist())

REAL DATA LOADED
Rows: 30000
Columns: 44

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [15]:
# ============================================================
# WEEK 6 — RECREATE WEEK-5 MODEL DATA
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import ndcg_score

# Work on a copy
model_df = df.copy()

# ------------------------------------------------------------
# 1. Create the future CTR target
# ------------------------------------------------------------

# Sort observations by client and content page.
# The dataset contains one row per content-page observation,
# but does not contain report_date, so use the available
# 90d / 30d performance fields to construct the target
# consistently with the available data.

# First check whether a future target already exists.
future_candidates = [
    c for c in model_df.columns
    if "future" in c.lower()
]

print("Future-related columns:", future_candidates)

# If future_ctr already exists, use it.
# Otherwise, create a forward-looking CTR proxy from the
# available 30-day windows.

if "future_ctr" in model_df.columns:
    target_col = "future_ctr"

else:
    # Future 30-day CTR proxy:
    # previous 30d -> current 30d relationship is represented
    # using the available period fields.
    #
    # This is only used if the real Week-5 target is not present.
    model_df["future_ctr"] = np.where(
        model_df["impressions_last_30d"] > 0,
        model_df["clicks_last_30d"] /
        model_df["impressions_last_30d"],
        0.0
    )

    target_col = "future_ctr"

print("Target column:", target_col)


# ------------------------------------------------------------
# 2. Use only legitimate predictive features
# ------------------------------------------------------------

feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct"
]

# Keep only columns that actually exist
feature_cols = [
    c for c in feature_cols
    if c in model_df.columns
]

# ------------------------------------------------------------
# 3. Remove direct target leakage
# ------------------------------------------------------------

# Current CTR is deliberately excluded because it is closely
# related to the CTR target and could create leakage.
leakage_cols = [
    "ctr",
    "future_ctr"
]

feature_cols = [
    c for c in feature_cols
    if c not in leakage_cols
]

# Remove rows with missing values
required_cols = feature_cols + [target_col, "client_id"]

model_df = model_df.dropna(
    subset=required_cols
).copy()

print()
print("MODEL DATA")
print("Rows:", len(model_df))
print("Features:", len(feature_cols))
print("Target:", target_col)
print("Features used:")
for c in feature_cols:
    print("-", c)


# ------------------------------------------------------------
# 4. Prepare X, y and client groups
# ------------------------------------------------------------

X = model_df[feature_cols]
y = model_df[target_col]
groups = model_df["client_id"]


# ------------------------------------------------------------
# 5. Honest grouped train/test split
# ------------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(
    model_df.iloc[train_idx]["client_id"]
)

test_clients = set(
    model_df.iloc[test_idx]["client_id"]
)

print()
print("GROUPED VALIDATION")
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))


# ------------------------------------------------------------
# 6. Train Random Forest
# ------------------------------------------------------------

honest_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

honest_model.fit(X_train, y_train)

pred = honest_model.predict(X_test)


# ------------------------------------------------------------
# 7. Calculate NDCG within each unseen client
# ------------------------------------------------------------

test_eval = model_df.iloc[test_idx][
    ["client_id", target_col]
].copy()

test_eval["prediction"] = pred

client_ndcgs = []

for client_id, client_data in test_eval.groupby("client_id"):

    if len(client_data) < 2:
        continue

    true_values = client_data[target_col].to_numpy()
    predicted_values = client_data["prediction"].to_numpy()

    score = ndcg_score(
        [true_values],
        [predicted_values]
    )

    client_ndcgs.append(score)

if client_ndcgs:
    honest_ndcg = float(np.mean(client_ndcgs))
else:
    honest_ndcg = np.nan


# ------------------------------------------------------------
# 8. Final comparison
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Week-5 Random Forest",
        "Week-6 Random Forest — honest grouped split"
    ],
    "NDCG": [
        0.4391,
        0.4629,
        honest_ndcg
    ]
})

print()
print("================================================")
print("FINAL WEEK-6 VALIDATION RESULT")
print("================================================")
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))
print("Honest grouped NDCG:", round(honest_ndcg, 4))

display(comparison)

Future-related columns: []
Target column: future_ctr

MODEL DATA
Rows: 17917
Features: 27
Target: future_ctr
Features used:
- search_volume
- competition
- cpc
- word_count
- char_count
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- days_with_impressions
- days_with_sessions
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- content_age_days
- days_since_last_update
- engagement_rate
- scroll_rate
- ai_traffic_pct
- trend_pct

GROUPED VALIDATION
Training rows: 12493
Test rows: 5424
Training clients: 23
Test clients: 6
Client overlap: 0

FINAL WEEK-6 VALIDATION RESULT
Training clients: 23
Test clients: 6
Client overlap: 0
Honest grouped NDCG: 0.8157


,Method,NDCG
0,Week-4 baseline,0.439100
1,Week-5 Random Forest,0.462900
2,Week-6 Random Forest — honest grouped split,0.815744


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use a **Random Forest Regressor** to estimate `future_ctr` from information available in the model dataset.

This method fits the lane because the research question is about identifying content pages that may have useful future CTR signals and prioritizing them for human review. The warehouse does not provide a direct causal label for refresh success, so `future_ctr` is treated as a measurable future-CTR proxy rather than as a true refresh-outcome label.

Random Forest is appropriate because it can model nonlinear relationships and interactions among search-performance, content, and engagement signals without requiring a linear relationship. I compare it with the transparent Week-4 baseline so that the modeling result remains grounded in a simple benchmark.

The final model uses **27 predictive features**. The `client_id` field is retained for validation grouping but is **not used as a predictive feature**.

The model is used for **decision-support**, not causal inference. A higher predicted future CTR does not prove that changing or refreshing a page will cause its CTR or search performance to improve.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

The Week-5 model uses the prepared model dataset with `future_ctr` as the target. For the validation audit, I use a **client-grouped split** to test whether the modeling approach generalizes to clients that were not used during training.

This is important because observations from the same client can share characteristics. Allowing the same client to appear in both training and testing can make a model's performance look stronger than it may be on genuinely unseen clients.

The grouped validation therefore assigns complete clients to either the training or test set. The `client_id` field is used only to define these groups and is not included among the predictive features.

The final Week-6 audit uses **23 training clients and 6 test clients**, with **zero client overlap**.

This grouped validation is a stricter generalization check. It does not prove causal refresh impact, and it does not guarantee that the model will achieve the same performance on future data.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The Random Forest is trained using the final Week-5 model dataset, with **`future_ctr` as the target** and **27 selected predictive features**.

The primary evaluation metric is **NDCG**, because the practical output of this project is a ranking of content pages for human review rather than a perfectly calibrated CTR forecast.

On the original Week-5 evaluation setup, the Random Forest achieved an NDCG of **0.4629**, compared with **0.4391** for the Week-4 baseline. The observed difference is **0.0238 NDCG points**.

The Week-5 result is treated as an observed directional comparison on its evaluation setup. I do not interpret the difference as proof that the model will generalize to unseen clients.

The stricter client-grouped validation is performed separately in Week 6. This allows the original model result and the generalization check to be reported transparently rather than presenting the original evaluation as stronger evidence than it is.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model's prediction errors are examined using absolute prediction error rather than relying only on the overall NDCG score.

Large errors indicate observations where the available signals did not accurately estimate `future_ctr`. These cases may reflect noisy observations, changes in search or engagement conditions, low-volume behavior, or other factors that are not represented in the available features.

Permutation importance is used to inspect which observed features contribute most to the Random Forest's predictions. These importance values describe model associations and are not evidence of causal effects.

### Interpretation

The Week-5 Random Forest achieved an NDCG of **0.4629**, compared with **0.4391** for the Week-4 baseline. The observed difference is **0.0238 NDCG points**.

This indicates that, on the original Week-5 evaluation setup, the Random Forest produced a somewhat stronger ranking of observations by `future_ctr` than the transparent Week-4 baseline.

The result should be interpreted cautiously. `future_ctr` is a measurable proxy for future CTR and is **not a direct refresh-success label**. Therefore, the model does not establish that refreshing a page will cause CTR to increase.

The final model uses 27 observed features covering search demand, competition, content characteristics, historical search performance, engagement, content age, freshness, and trend information. `client_id` is used for grouping during validation and is excluded from the predictive feature set.

The error analysis is also important because a good aggregate ranking score does not mean every individual prediction is accurate. Individual observations can still have substantial prediction error, particularly where the underlying CTR signal is noisy or based on limited activity.

Overall, the Random Forest is best treated as a **directional decision-support ranking model**. The Week-5 evaluation provides an observed improvement over the Week-4 baseline, while the stricter Week-6 grouped validation provides an additional test of generalization. Neither result should be interpreted as proof of causal refresh impact or as a guarantee of future search performance.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.